In [2]:
import pandas as pd
import json

## 数据读取，合并

In [2]:
comment_raw = pd.read_excel("/Users/clsljt/Documents/hhgh/station/data/XiZhan/XiZhan_mments.xlsx")
note_raw = pd.read_excel("/Users/clsljt/Documents/hhgh/station/data/XiZhan/XiZhan_notes.xlsx")

In [3]:
with open('/Users/clsljt/Documents/hhgh/station/data/XiZhan/xizhan_media_image_captions.json', 'r', encoding='utf-8') as file:
    fig_js = json.load(file)
fig_raw = pd.DataFrame([
    {
        "id": k.split("/")[0],
        "fig_name": k.split("/")[1],
        "content": v
    }
    for k, v in fig_js.items()
])

In [4]:
fig_raw.head()

,id,fig_name,content
0,679351350000000018013aa8,image_0.jpg,这张图片拍摄的是北京西站的正门入口区域，视角从道路正前方仰拍，展现了车站宏伟的建筑立面与周边...
1,67a72545000000001800409f,image_0.jpg,这张图片是一组拼贴图，旨在为乘坐地铁抵达北京西站的旅客提供一条“无需过安检”的快速进站路线攻...
2,67a72545000000001800409f,image_1.jpg,这张图片由上下两部分拼接而成，均拍摄于北京西站地铁站内，旨在为乘坐地铁抵达的旅客提供快速进站...
3,67a72545000000001800409f,image_2.jpg,这张图片拍摄于北京西站地铁站内，具体位置为“南广场”区域，场景为乘客出站后向上行进的过渡空间...
4,67a72545000000001800409f,image_3.jpg,这张图片拍摄于北京西站南广场区域，具体位置为地铁站出站后通往铁路进站口的过渡通道。画面呈现的...


In [5]:
# 图片描述按笔记合并，避免同一篇笔记的多张图片被当作多条独立舆情。
fig = (fig_raw.groupby("id", as_index=False)["content"]
       .agg(lambda values: " ".join(dict.fromkeys(str(v) for v in values if str(v).strip()))))
fig = fig.rename(columns={"content": "image_caption"})
# 多图笔记的自动描述很长，只保留前 1,200 字，避免模板性说明主导主题权重。
fig["image_caption"] = fig["image_caption"].str.slice(0, 1200)

note = note_raw[["note_id", "title", "desc", "tags"]].rename(columns={"note_id": "id"})
note = note.merge(fig, on="id", how="left")
comment = comment_raw[["note_id", "comment_id", "content"]].rename(columns={"note_id": "id"})

In [6]:
comment.head()

,id,comment_id,content
0,6a12f5ac000000000702babd,6a130af2000000002a004676,已经很多年不买到西站的票了，上周末没抢到去丰台的，只好买了西站。我去，出站口太臭了！感觉是污...
1,6a12f5ac000000000702babd,6a139b6b000000002900c5b6,去山西是吧[扯脸H]
2,6a12f5ac000000000702babd,6a13ae75000000002a033b41,嘿，聪明！
3,6a12f5ac000000000702babd,6a13afb300000000280360a1,山西是少见的可以选的[笑哭R]一般来说目的地和收发站就是绑定的，比如河北河南只有北京西，东北...
4,6a12f5ac000000000702babd,6a13b19d000000002901a0cd,好像还真是，不过河北其他地儿我不知道，保定是可以选的，北京几个站几乎都可以到，可能因为离北京...


In [7]:
note["content"] = (
    "标题：" + note["title"].fillna("").astype(str)
    + " 帖子内容：" + note["desc"].fillna("").astype(str)
    + " 包含标签：" + note["tags"].fillna("").astype(str)
    + " 图片内容：" + note["image_caption"].fillna("").astype(str)
)

# 保留笔记与评论；图片描述只作为所属笔记的补充语义，不单独进入情感筛选。
df = pd.concat([note[["id", "content"]], comment[["id", "content"]]], ignore_index=True)
df = df.dropna(subset=["content"]).drop_duplicates(subset=["id", "content"]).reset_index(drop=True)
print(f"笔记：{len(note)}，评论：{len(comment)}，待清洗文本：{len(df)}")

笔记：148，评论：5627，待清洗文本：5538


In [8]:
note.head()

,id,title,desc,tags,image_caption,content
0,6a12f5ac000000000702babd,北京西站难道不能整改一下么？,号称亚洲最大规模的铁路客运站之一的北京西站，地下车库脏乱差臭，都影响我们国家高铁的形象了，就...,['高铁出行'],这张图片拍摄于一个地下停车场的入口或通道区域，整体环境潮湿、地面有明显积水，光线来自顶部的荧...,标题：北京西站难道不能整改一下么？ 帖子内容：号称亚洲最大规模的铁路客运站之一的北京西站，地...
1,69798bd2000000000d008dbc,北京西站请你人性化一点,太离谱了北京西站，手受过伤不能提，候车室全程没有电梯，吃饭一楼是点餐二楼是吃饭，无障碍电梯是...,['北京西站没有电梯'],这张图片拍摄于一个大型铁路客运站的室内候车大厅，从空间结构、设施布局和人群活动来看，极有可能...,标题：北京西站请你人性化一点 帖子内容：太离谱了北京西站，手受过伤不能提，候车室全程没有电梯...
2,69675aca000000001a02aee2,9号线北京西站龟速问题什么时候才能解决,以前一直在说解决解决，解决这么久了也没见变快，反而变得更慢了[尬住R]\n#北京[话题]# ...,"['北京', '地铁', '北京地铁', '北京西站']",这张图片拍摄于北京地铁列车内部，视角为从车厢内向上仰拍，聚焦于车顶或车窗上方的线路指示牌。该...,标题：9号线北京西站龟速问题什么时候才能解决 帖子内容：以前一直在说解决解决，解决这么久了也...
3,68dc8b99000000000302dc23,是我的问题还是车站设置问题啊,每次北京西站出了检票口都是1车厢和15车厢，横跨整个车才能走到自己的车厢，是不是车站设置有问...,"['北京西站', '高铁', '高铁出行']",这张图片拍摄于一个现代化的铁路车站站台，视角从站台边缘或列车内部向外拍摄，呈现出清晰的层次感...,标题：是我的问题还是车站设置问题啊 帖子内容：每次北京西站出了检票口都是1车厢和15车厢，横...
4,689af376000000001d007a41,北京西站英文名称,火车站名中的方位词是车站名的一部分，不应该单拿出来翻译。\n#翻译[话题]# #有趣的地名[...,"['翻译', '有趣的地名']",这张图片由两张照片和一段文字说明构成，整体呈现为一个图文并茂的提问式内容，旨在探讨北京西站英...,标题：北京西站英文名称 帖子内容：火车站名中的方位词是车站名的一部分，不应该单拿出来翻译。\...


In [11]:
len(df)

5538

### negative csv read

In [44]:
import ast
import json
from pathlib import Path

import pandas as pd

csv_path = Path("output/0623/xizhan_negative_texts.csv")
source_df = pd.read_csv(csv_path, dtype={"id": str}, keep_default_na=False, encoding="utf-8-sig")


def unpack_texts(value, is_list: bool) -> list[str]:
    """Return non-empty texts from either a scalar cell or a JSON list cell."""
    if value is None or pd.isna(value) or not str(value).strip():
        return []
    if not is_list:
        return [str(value).strip()]
    try:
        items = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        # Compatible with legacy Python-list strings if the CSV is regenerated differently.
        items = ast.literal_eval(value)
    return [str(item).strip() for item in items if str(item).strip()]


records = []
for form, is_list in [("title", False), ("content", False), ("comment", True), ("fig", True)]:
    for note_id, value in source_df[["id", form]].itertuples(index=False):
        records.extend(
            {"id": note_id, "content": text, "form": form}
            for text in unpack_texts(value, is_list)
        )

# 最终逐文本数据：id 为原 CSV 的笔记 ID；form 标识文本来源。
df = pd.DataFrame(records, columns=["id", "content", "form"]).drop_duplicates().reset_index(drop=True)
df

,id,content,form
0,685a71b400000000220305eb,无标题,title
1,6896576d000000002502bd1c,无标题,title
2,68e6921f000000000300edfa,无标题,title
3,68fe0d30000000000303b447,北京西站真的不能有交通管理员吗?,title
4,6977141d000000002203162e,非京户籍人禁止违法代开北京巡回出租车,title
...,...,...,...
477,68fe0d30000000000303b447,"这张图片是一张以文字为核心的视觉设计,其内容并非对真实车站空间的现场记录,而是对北京西站交通...",fig
478,6977141d000000002203162e,"这张图片拍摄于一辆正在运行的出租车内部,视角从乘客座位朝向车辆前部,聚焦于仪表盘区域。整体环...",fig
479,698edca6000000001a030643,"这张图片拍摄于一个城市户外的非正式公共空间,整体场景呈现为一个略显破败、功能性的街角或巷道边...",fig
480,6a12f5ac000000000702babd,"这张图片拍摄于一个地下停车场的入口或通道区域,整体环境潮湿、地面有明显积水,光线来自顶部的荧...",fig


### LLM Describe

In [4]:
BATCH_SIZE = 16
MIN_TEXT_LENGTH = 4
N_WORDS = 10
N_TOPICS = 8
with open("data/XiZhan/xizhan_analysis.json", "r", encoding="utf-8") as f:
    rawjs = json.load(f)

raw = pd.DataFrame(rawjs).rename(columns={"description": "content"})

# 仅聚类 business 类别，并保留与 documents 一一对应的源数据。
source_df = raw.loc[raw["category_group"].eq("business"), ["note_id", "content"]].copy()
clean_content = source_df["content"].fillna("").astype(str).str.strip()
keep = clean_content.ne("") & clean_content.ne("无标题") & clean_content.str.len().ge(MIN_TEXT_LENGTH)
result_df = source_df.loc[keep].rename(columns={"note_id": "id"}).reset_index(drop=True)
documents = clean_content.loc[keep].tolist()
result_df["content"] = documents

if len(documents) < N_TOPICS:
    raise ValueError(f"可聚类文本仅有 {len(documents)} 条，少于 N_TOPICS={N_TOPICS}")

print(f"待聚类文本：{len(documents):,} 条")
raw.head()



待聚类文本：48 条


,note_id,title,upload_time,liked_count,comment_count,content,category,category_group,raw_response,num_images,collected_count,share_count
0,679351350000000018013aa8,北京西站上面的大亭子可以上去参观吗,2025-01-24 16:37:09,2069,290,用户询问北京西站顶部标志性建筑（大亭子）是否开放参观，并表达了想登高俯瞰广场的意愿。评论区内...,xhs_other,xhs,"```json\n{\n ""description"": ""用户询问北京西站顶部标志性建筑（...",1,506,284
1,67a72545000000001800409f,不用安检！北京西站3分钟进站（地铁到达）,2025-02-08 18:35:05,4561,168,该帖分享了北京西站地铁到达的快速进站攻略。作者指出，乘坐地铁到达的旅客可通过南广场或北广场的...,xhs_guide,xhs,"```json\n{\n ""description"": ""该帖分享了北京西站地铁到达的快速...",4,4214,2726
2,67c90fe500000000060299c4,北京西站的迷惑设计,2025-03-06 11:00:53,22,27,用户吐槽北京西站存在“迷惑设计”，导致南北广场之间通行困难。虽然导航显示直线距离近，但实际需...,navigation_guidance,business,"```json\n{\n ""description"": ""用户吐槽北京西站存在“迷惑设计”...",1,4,6
3,68121219000000002001fd02,北京西站刚刚怎么了,2025-04-30 20:05:45,348,103,用户分享了在北京西站G897检票口目睹的惊险一幕：一名白衣男子试图强行闯关，撞开路人并拖拽女...,safety_emergency,business,"```json\n{\n ""description"": ""用户分享了在北京西站G897检票...",2,49,233
4,6839977b000000000c038b8d,北京西你要不赔我点钱吧,2025-05-30 19:33:15,34,31,用户吐槽在北京西站遭遇高铁严重晚点，导致行程受阻。尽管特意购买了始发车次（G555），仍因前...,digital_information,business,"```json\n{\n ""description"": ""用户吐槽在北京西站遭遇高铁严重晚...",2,5,8


## 清洗
- 对 text 做基础清洗：去空白、去 URL、去 @、去小红书表情、保留话题词、删除纯标点和明显闲聊文本

In [12]:
import sys
sys.path.insert(0, "/Users/clsljt/Documents/hhgh/station/scripts")

from clean_station_data import clean_text_id_dataframe

In [13]:
cleaned_df = clean_text_id_dataframe(df)

In [14]:
cleaned_df.head()

,id,content
0,6a12f5ac000000000702babd,"标题:北京西站难道不能整改一下么? 帖子内容:号称亚洲最大规模的铁路客运站之一的北京西站,地..."
1,69798bd2000000000d008dbc,"标题:北京西站请你人性化一点 帖子内容:太离谱了北京西站,手受过伤不能提,候车室全程没有电梯..."
2,69675aca000000001a02aee2,"标题:9号线北京西站龟速问题什么时候才能解决 帖子内容:以前一直在说解决解决,解决这么久了也..."
3,68dc8b99000000000302dc23,"标题:是我的问题还是车站设置问题啊 帖子内容:每次北京西站出了检票口都是1车厢和15车厢,横..."
4,689af376000000001d007a41,"标题:北京西站英文名称 帖子内容:火车站名中的方位词是车站名的一部分,不应该单拿出来翻译。 ..."


In [15]:
len(cleaned_df)

5192

## 情感分析

In [16]:
from sentiment_filter import filter_neutral_negative

In [17]:
# 设置阈值
# filtered_df = filter_neutral_negative(
#     cleaned_df,
#     negative_threshold=0.55,
#     positive_threshold=0.65
# )
# 
# 只返回两列
# filtered_df = filter_neutral_negative(cleaned_df, keep_scores=False)

In [18]:
filtered_df= filter_neutral_negative(cleaned_df)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /Users/clsljt/Documents/hhgh/station/senModel
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment:   0%|          | 0/163 [00:00<?, ?it/s]

In [19]:
len(filtered_df)


2324

In [20]:
filtered_df.head()

,id,content,sentiment,p_negative,p_positive
0,6a12f5ac000000000702babd,"标题:北京西站难道不能整改一下么? 帖子内容:号称亚洲最大规模的铁路客运站之一的北京西站,地...",negative,0.998362,0.001638
1,69798bd2000000000d008dbc,"标题:北京西站请你人性化一点 帖子内容:太离谱了北京西站,手受过伤不能提,候车室全程没有电梯...",negative,0.997568,0.002432
2,689af376000000001d007a41,"标题:北京西站英文名称 帖子内容:火车站名中的方位词是车站名的一部分,不应该单拿出来翻译。 ...",neutral,0.432826,0.567174
3,6839977b000000000c038b8d,"标题:北京西你要不赔我点钱吧 帖子内容:说真的我都没见过这么离谱的事,四年前我在北京西坐高铁...",negative,0.907627,0.092373
4,6989ed29000000001a02b3ef,"标题:无标题 帖子内容:地铁过去的,出站后上广场就直接进站,还想着马上安检,谁知已经到候车大...",negative,0.989206,0.010794


## 职责标注（不参与前置过滤）

这里只给文本增加职责类别、问题信号和整改候选字段；BERTopic 仍使用情感筛选后的完整文本。主题生成后，再用这些字段判断哪些主题值得进入报告。


In [19]:
from clean_station_data import annotate_text_id_dataframe, build_rectification_ledger

# 只做标注，不在 BERTopic 前过滤。
# BERTopic 使用情感筛选后的完整语料；职责相关和整改信号留到主题生成后解释。
filtered_df = annotate_text_id_dataframe(filtered_df)
rectification_ledger = build_rectification_ledger(filtered_df)

print("情感筛选后文本数:", len(filtered_df))
print("宽松职责相关文本数:", int(filtered_df["is_responsibility_relevant"].sum()))
print("明确整改候选文本数:", int(filtered_df["rectification_ready"].sum()))
rectification_ledger


情感筛选后文本数: 2324
宽松职责相关文本数: 473
明确整改候选文本数: 140


,primary_issue_category,evidence_count,like_sum,severity_mix,responsible_owner,suggested_action,representative_evidence,representative_urls
0,人员服务,41,0,中|高,客运服务/投诉处置,复盘服务态度、解释口径和投诉闭环，补齐现场响应机制。,"我在和你说语言范畴的事情,你一直在扯其他的东西。地理区位是既定的约定俗成的,场站作为功能性建...",
8,电梯无障碍,32,0,中|高,设施设备/客运服务；客运组织/动线管理；导视系统/客运服务；客流组织/现场秩序；候车服务/商...,核查扶梯直梯开放、故障、导引和大件行李通行体验。；复核高频路径的导流、检票安检排队和换乘衔接...,"标题:北京西站过夜指南 帖子内容:在鸟巢看完演唱会买了第二天早上六点二十的高铁,为了省一晚住...",
7,环境卫生,19,0,中|高,环境卫生/物业运维|环境卫生/物业运维；设施设备/物业运维；设施设备/客运服务；客运组织/动...,排查异味、保洁、通风和垃圾清运薄弱点，形成点位巡检清单。,"标题:北京西站难道不能整改一下么? 帖子内容:号称亚洲最大规模的铁路客运站之一的北京西站,地...",
2,商业价格,12,0,低|中|高,商业管理/价格监管协同|商业管理/价格监管协同；客运服务/投诉处置,梳理价格投诉点，核查明码标价和高频商品服务价格。,"笑死我了不简单就不干了?还不让说。你学建筑的你去弄呗。又不弄,又不让语言类专业的说,人家就学...",
9,进出站动线,12,0,中|低,客运组织/动线管理；交通接驳/停车管理；客流组织/现场秩序|客运组织/动线管理,复核高频路径的导流、检票安检排队和换乘衔接。；优化网约车、出租车、自驾接送点位和停车场秩序提...,标题:北京最拥堵十条地铁线 你每天挤哪几条? 帖子内容:以下北京地铁线路在客流量、断面满载率...,
3,拥挤排队,10,0,中,客流组织/现场秩序|客流组织/现场秩序；候车服务/商业配套,识别高峰拥堵环节，调整开口、排队蛇形线和现场疏导。,"最无语的那么大个站台,那么长一条!无论是哪个车厢候车排队点都得被迫闻烟味......就为了烟...",
5,标识指引,7,0,低,导视系统/客运服务|导视系统/客运服务；交通接驳/停车管理,补强易迷路点位的标识、地贴、电子屏和人工引导。,"我有一年出差也干过这事儿,北京站的车我到了西站死活找不到这个车次,我还纳闷呢,到窗口一问,人...",
4,接送停车,4,0,低|中|高,交通接驳/停车管理|交通接驳/停车管理；客流组织/现场秩序,优化网约车、出租车、自驾接送点位和停车场秩序提示。,"打12328举报 就是这帮人搞的大家都说北京出租车不好 || 是8号晚上吧,不是车难打,是打...",
6,漏水积水,2,0,中,设施设备/物业运维；客运组织/动线管理；导视系统/客运服务；客流组织/现场秩序；候车服务/商...,核查漏水积水点位，区分临修、排水、结构维修并跟踪闭环。；复核高频路径的导流、检票安检排队和换...,"标题:注意啦!尽量不走北京西站北广场 帖子内容:今天乘坐高铁,不小心从北广场出站,结果出来后...",
1,候车配套,1,0,低,候车服务/商业配套,评估候车座椅、充电、夜间服务、餐饮和寄存供给。,"不能,只能休息",


## 词频分析

In [6]:
from scripts.word_frequency import compute_word_frequency, generate_wordcloud_from_frequency

In [7]:

freq_df = compute_word_frequency(
    raw,
    use_stopinthis=True
)


/opt/anaconda3/envs/py312/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/4y/vlf2jr0n0bgc5_572sb76tzw0000gn/T/jieba.cache
Loading model cost 0.376 seconds.
Prefix dict has been built successfully.


In [22]:
freq_df.head(50)

,word,count,doc_count,frequency
0,站台,166,69,0.008456
1,列车,105,42,0.005349
2,标识,98,46,0.004992
3,吸烟,91,55,0.004636
4,候车,90,40,0.004585
5,地铁,75,50,0.003820
6,广场,68,30,0.003464
7,环境,64,32,0.003260
8,进站,62,39,0.003158
9,闸机,60,37,0.003056


In [8]:
generate_wordcloud_from_frequency(
    freq_df.head(200),
    "/Users/clsljt/Documents/hhgh/station/outputs/wordcloud.png"
)

PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/wordcloud.png')

## BERTopic 建模与主题后筛选


In [5]:
from scripts.run_bertopic import (
    run_bertopic_on_filtered_df,
)

- min_topic_size=10：一个主题至少要包含 10 条文本。少于 10 条的聚类通常会被归为离群点（-1）或无法形成稳定主题。数值越大，主题越少、越稳定；越小，主题越多、越细碎。

- nr_topics=None：不强制压缩主题数量，保留模型初始识别出的主题数。  
    - 设为整数，例如 nr_topics=20：将主题进一步合并，最终尽量得到约 20 个主题。
    - 设为 "auto"：由 BERTopic 自动合并相似主题。
    - None：不额外合并。

In [19]:
bt = run_bertopic_on_filtered_df(
    df[df["form"] != "fig"],
    min_topic_size=5,
    nr_topics=None,  # 保留细粒度主题，后续按质量规则筛选
    run_name="station_filtered_bertopic",
    responsibility_relevant_only=False
)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

2026-06-24 10:14:44,609 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

2026-06-24 10:14:45,663 - BERTopic - Embedding - Completed ✓
2026-06-24 10:14:45,663 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-24 10:14:46,009 - BERTopic - Dimensionality - Completed ✓
2026-06-24 10:14:46,010 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-24 10:14:46,023 - BERTopic - Cluster - Completed ✓
2026-06-24 10:14:46,024 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-24 10:14:46,062 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00, 1060.51it/s]
2026-06-24 10:14:46,101 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-24 10:14:46,117 - BERTopic - WARNING: When you use `pic

In [20]:
bt.topic_info

,Topic,Count,Name,Representation,Representative_Docs,assigned_rows
0,-1,1,-1_不是带了个防晒吗___,"[不是带了个防晒吗, , , , , , , , , ]","[不是带了个防晒吗, 你这不抬杠吗,要不大家都不穿衣服了?]",1
1,0,269,0_北京西站_10号线_地铁_14号线,"[北京西站, 10号线, 地铁, 14号线, 5号线, 没用, 没有, 8号线, 不能, 6号线]","[太离谱了北京西站,手受过伤不能提,候车室全程没有电梯,吃饭一楼是点餐二楼是吃饭,无障碍电梯...",269
2,1,164,1_衣服上_坐火车回家_20都很有效果了_没人在意你们感受只是单纯想吸烟,"[衣服上, 坐火车回家, 20都很有效果了, 没人在意你们感受只是单纯想吸烟, 没人不就没人...",[我觉得要不就一开始都收了 要不就搞一个吸烟区 把吸烟的都赶进去 只要有吸烟区 有人在外面吸...,164
3,2,24,2_中距离短距离坐飞机的时间比高铁长多了_高铁禁止超量携带指甲油是因它是高度易燃的液体危险品...,"[中距离短距离坐飞机的时间比高铁长多了, 高铁禁止超量携带指甲油是因它是高度易燃的液体危险品...",[飞机只有在长距离才快 中距离短距离坐飞机的时间比高铁长多了 提前一两个小时去机场到地方还要...,24
4,3,13,3_广州吸烟南站_香港吸烟西九龙站_成都吸烟东站_上海吸烟虹桥站,"[广州吸烟南站, 香港吸烟西九龙站, 成都吸烟东站, 上海吸烟虹桥站, 上海吸烟站, 上海吸...","[成都吸烟东站, 广州吸烟南站 深圳吸烟北站 香港吸烟西九龙站, 苏州吸烟站 苏州吸烟南站 ...",13


In [43]:
k = 0
bt.assignments[bt.assignments["topic"] == 2][["content"]][k:k+30]

,content
138,"但是谁出钱呢 为啥大家纳的税要用来干这个呢,直接禁止带打火机不就好了..."
200,安检不是不让带打火机吗?打火机怎么带进去的
201,飞机不让带火机高铁不管
218,"我想知道烟和打火机是怎么带上高铁的,安检的时候给过吗?"
219,高铁不管的
225,"坐飞机嘛,还快些"
226,飞机只有在长距离才快 中距离短距离坐飞机的时间比高铁长多了 提前一两个小时去机场到地方还要滑...
235,最好的就是不允许带火机上高铁
239,优先选择飞机
256,"支持⬆️二手烟危害比一手烟大,而且不明白为啥火车高铁可以带打火机"


In [45]:
bt.assignments.to_csv("bertopic.csv")

## 手工选择 Topic 并渲染可视化

在下方 `MANUAL_TOPICS` 填入需要展示的 topic ID，例如 `[5, 9, 13]`。保留空列表 `[]` 时渲染全部非离群 topic；同一列表会用于柱状图、二维主题分布、热力图、层级图和词项排名图。

In [31]:
from pathlib import Path

# 手动填写要展示的 topic ID；空列表 [] 表示展示全部非离群 topic。
# MANUAL_TOPICS = [i for i in range(35) if i not in []]  # 例如：[5, 9, 13]
MANUAL_TOPICS = [i for i in range(4)]  # 例如：[5, 9, 13]
manual_vis_dir = Path("/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations")
# 每个主题在图中显示和导出的关键词数。
# 8 个最适合报告阅读；需要更多词时可改为 30 或 50。
KEYWORDS_PER_TOPIC = 8

# 仅更新主题的关键词表示，不重新聚类，因此 topic ID 和每条文本的归属不会改变。
if bt.topic_model.top_n_words < KEYWORDS_PER_TOPIC:
    bt.topic_model.update_topics(
        bt.assignments["content"].tolist(),
        top_n_words=KEYWORDS_PER_TOPIC,
        vectorizer_model=bt.topic_model.vectorizer_model,
        ctfidf_model=bt.topic_model.ctfidf_model,
    )
    bt.topic_info = bt.topic_model.get_topic_info()

available_topics = bt.topic_info.loc[bt.topic_info["Topic"].ne(-1), "Topic"].astype(int).tolist()
render_topics = available_topics if not MANUAL_TOPICS else [topic for topic in MANUAL_TOPICS if topic in available_topics]
missing_topics = sorted(set(MANUAL_TOPICS) - set(available_topics))

if missing_topics:
    print(f"忽略当前模型中不存在的 topic: {missing_topics}")
if not render_topics:
    raise ValueError("没有可渲染的 topic；请检查 MANUAL_TOPICS。")

manual_vis_dir.mkdir(parents=True, exist_ok=True)
manual_visualization_paths = {}

# 逐词导出，便于查看每个主题的前 KEYWORDS_PER_TOPIC 个关键词和 c-TF-IDF 权重。
keyword_rows = []
for topic in render_topics:
    for rank, (keyword, score) in enumerate((bt.topic_model.get_topic(topic) or [])[:KEYWORDS_PER_TOPIC], start=1):
        keyword_rows.append({"Topic": topic, "rank": rank, "keyword": keyword, "c_tf_idf": score})
manual_keywords_path = manual_vis_dir / "manual_topic_keywords.csv"
pd.DataFrame(keyword_rows).to_csv(manual_keywords_path, index=False, encoding="utf-8-sig")
manual_visualization_paths["keywords_csv"] = manual_keywords_path

manual_barchart = bt.topic_model.visualize_barchart(
    topics=render_topics, top_n_topics=None, n_words=KEYWORDS_PER_TOPIC,
    width=600, height=360, autoscale=True
)
manual_visualization_paths["barchart"] = manual_vis_dir / "barchart_manual_topics.html"
manual_barchart.write_html(str(manual_visualization_paths["barchart"]))

# 多主题拼图用于概览；同时给每个主题单独导出一张宽图，避免长关键词被相邻面板遮挡。
single_barchart_dir = manual_vis_dir / "individual_barcharts"
single_barchart_dir.mkdir(exist_ok=True)
for topic in render_topics:
    single_barchart = bt.topic_model.visualize_barchart(
        topics=[topic], top_n_topics=None, n_words=KEYWORDS_PER_TOPIC,
        title=f"Topic {topic} — Top {KEYWORDS_PER_TOPIC} keywords",
        width=900, height=420, autoscale=True
    )
    single_path = single_barchart_dir / f"barchart_topic_{topic}.html"
    single_barchart.write_html(str(single_path))
    manual_visualization_paths[f"barchart_topic_{topic}"] = single_path

manual_topics_map = bt.topic_model.visualize_topics(
    topics=render_topics, top_n_topics=None
)
manual_visualization_paths["topics"] = manual_vis_dir / "topics_manual_topics.html"
manual_topics_map.write_html(str(manual_visualization_paths["topics"]))

manual_heatmap = bt.topic_model.visualize_heatmap(
    topics=render_topics, top_n_topics=None
)
manual_visualization_paths["heatmap"] = manual_vis_dir / "heatmap_manual_topics.html"
manual_heatmap.write_html(str(manual_visualization_paths["heatmap"]))

manual_hierarchy = bt.topic_model.visualize_hierarchy(
    topics=render_topics, top_n_topics=None
)
manual_visualization_paths["hierarchy"] = manual_vis_dir / "hierarchy_manual_topics.html"
manual_hierarchy.write_html(str(manual_visualization_paths["hierarchy"]))

manual_term_rank = bt.topic_model.visualize_term_rank(topics=render_topics)
manual_visualization_paths["term_rank"] = manual_vis_dir / "term_rank_manual_topics.html"
manual_term_rank.write_html(str(manual_visualization_paths["term_rank"]))

print(f"渲染 topic（{len(render_topics)} 个）: {render_topics}")
manual_visualization_paths

渲染 topic（4 个）: [0, 1, 2, 3]


{'keywords_csv': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/manual_topic_keywords.csv'),
 'barchart': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/barchart_manual_topics.html'),
 'barchart_topic_0': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/individual_barcharts/barchart_topic_0.html'),
 'barchart_topic_1': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/individual_barcharts/barchart_topic_1.html'),
 'barchart_topic_2': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/individual_barcharts/barchart_topic_2.html'),
 'barchart_topic_3': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs/report_0623/manual_topic_visualizations/individual_barcharts/barchart_topic_3.html'),
 'topics': PosixPath('/Users/clsljt/Documents/hhgh/station/outputs